# TCN unit commitment

**TDT4173 Modern Machine Learning in Practice, Group 61**  
**Authors:** Fredrik Nymoen (106988) and Jonas Birkeli (107006)  
**Kaggle team:** [61] Nymoen Birkeli

Run all cells in a fresh Python kernel. This notebook reads only the supplied raw
course files, trains from scratch on all 2,922 labeled cases from 2015 to 2022,
and predicts the 725 cases from 2023 to 2024. Each case contains 168 hours and
14 generators. No installation, downloads, saved models or other notebooks are used.
Dependencies are pinned in `requirements_submission.txt` (Python 3.14.7).

Place `data/kernel/`, `data/extended/`, `prediction_mapping.csv` and `sample_submission.csv` in the same folder as this notebook.

One shared temporal network jointly predicts all 14 generators. It uses
913 hourly and 182 static inputs and has 368,506 parameters.
The output is `outputs/tcn_submission.csv`.


## Fixed training settings

Train for 22 epochs with AdamW, learning rate 0.0005 and batch size 8.
The learning rate follows a cosine schedule defined over 100 epochs,
with training ending at epoch 22.
Seven semantic input encoders and nine local plant branches learn together
inside one shared network. This is a standalone model. The report explains
the feature and parameter experiments behind these fixed choices.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import platform
import re
import shutil
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "data/kernel/Unit_commitment_decisions.csv").is_file()), Path.cwd())
HOURS = 168
SEED = 42
THREADS = 4
FEATURE_VERSION = "hydro-decision-v1"
DECISION_GROUPS = {"economics", "water_balance"}
NOTEBOOK_STARTED = perf_counter()

import random

BASE_PARAMETERS = {'width': 64,
 'dropout': 0.1,
 'dilations': [1, 2, 4, 8, 16, 32, 64],
 'kernel_size': 3,
 'batch_size': 8,
 'learning_rate': 0.0005,
 'weight_decay': 0.0001,
 'max_epochs': 100,
 'gradient_clip': 1.0,
 'normalization_clip': 10.0,
 'lr_schedule': 'cosine_epoch_max_epochs',
 'device': 'cpu',
 'input_encoding': 'asinh',
 'regime_features': True,
 'merit_features': False,
 'allocation_step_hours': 6,
 'history_features': False,
 'head_width': 0,
 'environment_features': True,
 'dual_features': True,
 'compact_features': False,
 'dual_context_features': True,
 'bypass_scenario_features': True,
 'river_features': True,
 'commitment_dp_features': True,
 'local_head_width': 16,
 'local_input_mode': 'plant',
 'local_hydraulic_routing': True,
 'documented_terminal_conversion': True,
 'allocation_precompute_workers': 4,
 'layout': 'shared',
 'typed_encoder_width': 16}
EPOCHS = 22
FEATURE_SET = "allocation_physics"
PHYSICS_GROUPS = {"hydraulics", "economics", "network"}
OUTPUT_PATH = ROOT / "outputs/tcn_submission.csv"
EXPECTED_CSV_SHA256 = "84a83f1df1206acfbb69b32c9ec831e4b12fa7c6c64affd9ac1bcac88613c693"
print("Python:", platform.python_version())
print({name: metadata.version(name) for name in ("torch", "numpy", "pandas", "scipy", "PyYAML")})
display(pd.DataFrame({"setting": ["epochs", "batch size", "learning rate", "cosine denominator", "CPU threads"],
                      "value": [EPOCHS, BASE_PARAMETERS["batch_size"], BASE_PARAMETERS["learning_rate"],
                                BASE_PARAMETERS["max_epochs"], THREADS]}))


Python: 3.14.7
{'torch': '2.14.1+cpu', 'numpy': '2.5.3', 'pandas': '3.0.6', 'scipy': '1.18.1', 'PyYAML': '6.0.3'}


,setting,value
0,epochs,22.0000
1,batch size,8.0000
2,learning rate,0.0005
3,cosine denominator,100.0000
4,CPU threads,4.0000


## Raw data and hydraulic topology

Only the supplied data are used. Timestamps are UTC and object names define
all hydraulic connections. Storage is observed only at case start, water values
are taken at the week end, and exogenous prices/inflow/constraints cover the
168-hour horizon.


In [2]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")


2922 training cases; 725 test cases; 2352 targets


In [3]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))
# Directed hydraulic edges exclude generator membership edges. Stop at the first
# receiving reservoir; never interpret graph connectivity as realized routing.
HYDRAULIC_KINDS = {"reservoir", "plant", "river", "tunnel"}
FORWARD, BACKWARD = {}, {}
for edge in connections:
    if edge["from_type"] in HYDRAULIC_KINDS and edge["to_type"] in HYDRAULIC_KINDS:
        left = (edge["from_type"], edge["from"])
        right = (edge["to_type"], edge["to"])
        FORWARD.setdefault(left, set()).add(right)
        BACKWARD.setdefault(right, set()).add(left)
PLANT_FOR = {g: next(e["to"] for e in connections if e["from"] == g
                     and e["from_type"] == "generator" and e["to_type"] == "plant")
             for g in GENERATORS}
PLANT_GENERATORS = {p: [g for g in GENERATORS if PLANT_FOR[g] == p]
                    for p in topology["model"]["plant"]}
PLANT_FEEDERS = {p: FEEDERS[gens[0]] for p, gens in PLANT_GENERATORS.items()}

def reachable(start, adjacency, stop_at_reservoir=False):
    pending, seen = list(adjacency.get(start, [])), set()
    while pending:
        node = pending.pop()
        if node == start or node in seen:
            continue
        seen.add(node)
        if not (stop_at_reservoir and node[0] == "reservoir"):
            pending.extend(adjacency.get(node, []))
    return seen

RECEIVERS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD, True)
                       if kind == "reservoir") for p in PLANT_GENERATORS}
UPSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), BACKWARD)
                             if kind == "plant") for p in PLANT_GENERATORS}
DOWNSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD)
                               if kind == "plant") for p in PLANT_GENERATORS}
LEVEL_CURVES = {r: (np.asarray(topology["model"]["reservoir"][r]["vol_head"]["x"], dtype=float),
                    np.asarray(topology["model"]["reservoir"][r]["vol_head"]["y"], dtype=float))
                for r in RESERVOIRS}
for r, (x, y) in LEVEL_CURVES.items():
    if len(x) != len(y) or not np.all(np.diff(x) > 0) or not np.all(np.diff(y) >= 0):
        raise ValueError(f"Invalid volume-level curve: {r}")
GEN_SPECS = topology["model"]["generator"]


## Economic context and LP water allocation

Price/value comparisons and water-balance summaries describe each intake.
The continuous six-hour LP routes water through the supplied system and
projects storage and allocation. These projections are constructed from
available inputs and do not use future observed storage or UC labels.


In [4]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32)


In [5]:
ALLOCATION_STEP_HOURS = BASE_PARAMETERS["allocation_step_hours"]
ALLOCATION_CACHE = {}
ALLOCATION_STRUCTURE = None

def efficiency(spec, head, power):
    head=np.asarray(head,dtype=np.float64)
    curves=sorted(spec["turb_eff_curves"],key=lambda c:c["ref"])
    refs=np.array([c["ref"] for c in curves])
    q=power*1e6/(1000*9.81*head*0.8)
    generator_curve=spec["gen_eff_curve"]
    eta_gen=np.interp(power,generator_curve["x"],np.array(generator_curve["y"])/100)
    for _ in range(4):
        etas=np.stack([np.interp(q,c["x"],np.array(c["y"])/100) for c in curves],axis=-1)
        upper=np.clip(np.searchsorted(refs,head,side="right"),1,max(1,len(refs)-1))
        if len(refs)==1:
            eta_turb=etas[...,0]
        else:
            lower=upper-1
            fraction=np.clip((head-refs[lower])/(refs[upper]-refs[lower]),0,1)
            eta_turb=np.take_along_axis(etas,lower[...,None],axis=-1)[...,0]*(1-fraction)+np.take_along_axis(etas,upper[...,None],axis=-1)[...,0]*fraction
        eta=np.clip(eta_gen*eta_turb,0.3,1.0)
        q=power*1e6/(1000*9.81*head*eta)
    return q,eta


def allocation_structure():
    """Build a sparse conservative-flow relaxation of the supplied directed graph."""
    from scipy.sparse import lil_matrix
    plants = list(PLANT_GENERATORS)
    nodes = sorted(set(FORWARD) | {node for values in FORWARD.values() for node in values})
    arcs = sorted((left, right) for left, values in FORWARD.items() for right in values)
    # Terminal river reaches and Hogga discharge into the external sink.
    arcs += [(node, ("sink", "outside")) for node in nodes if not FORWARD.get(node)]
    count = HOURS // ALLOCATION_STEP_HOURS
    nf, nr, nn = len(arcs), len(RESERVOIRS), len(nodes)
    node_position = {node: index for index, node in enumerate(nodes)}
    nv = count * (nf + 3 * nr)
    equality = lil_matrix((count * nn, nv), dtype=np.float64)
    inequality = lil_matrix((2 * count * nr, nv), dtype=np.float64)
    for step in range(count):
        for edge, (left, right) in enumerate(arcs):
            equality[step * nn + node_position[left], step * nf + edge] += 1
            if right in node_position:
                equality[step * nn + node_position[right], step * nf + edge] -= 1
        for r, reservoir in enumerate(RESERVOIRS):
            column = count * nf + step * nr + r
            row = step * nn + node_position[("reservoir", reservoir)]
            equality[row, column] += 1
            if step:
                equality[row, column - nr] -= 1
            # Soft upper/lower storage constraints keep exceptional raw cases
            # feasible; their violations are exposed as features, not concealed.
            upper, lower = 2 * (step * nr + r), 2 * (step * nr + r) + 1
            inequality[upper, column] = 1
            inequality[upper, count * (nf + nr) + step * nr + r] = -1
            inequality[lower, column] = -1
            inequality[lower, count * (nf + 2 * nr) + step * nr + r] = -1
    return {"plants": plants, "nodes": nodes, "arcs": arcs, "steps": count,
            "nf": nf, "nr": nr, "nn": nn, "nv": nv,
            "node_position": node_position, "eq": equality.tocsr(), "ub": inequality.tocsr()}


def allocation_case(start):
    """Use only S state, supplied planning inputs and S+168h values; never UC labels."""
    from scipy.optimize import linprog
    import warnings
    start = pd.Timestamp(start)
    if start in ALLOCATION_CACHE:
        return ALLOCATION_CACHE[start]
    global ALLOCATION_STRUCTURE
    if ALLOCATION_STRUCTURE is None:
        ALLOCATION_STRUCTURE = allocation_structure()
    s = ALLOCATION_STRUCTURE
    k, nf, nr, nn = s["steps"], s["nf"], s["nr"], s["nn"]
    times = pd.date_range(start, periods=HOURS, freq="h")
    volume = tables["volumes"].loc[start, RESERVOIRS].to_numpy(dtype=float)
    terminal = tables["water_values"].loc[start + pd.Timedelta(hours=HOURS), RESERVOIRS].to_numpy(dtype=float)
    price = tables["prices"].reindex(times).iloc[:, 0].to_numpy(dtype=float).reshape(k, -1).mean(1)
    inflow = tables["inflows"].reindex(times)
    minvol = tables["min_volume"].reindex(times)
    minflow = tables["min_flow"].reindex(times)
    seconds = ALLOCATION_STEP_HOURS * 3600
    qmax, factor = {}, {}
    levels = {r: np.interp(volume[i], *LEVEL_CURVES[r]) for i, r in enumerate(RESERVOIRS)}
    for plant in s["plants"]:
        feeders = PLANT_FEEDERS[plant]
        level = np.average([levels[r] for r in feeders], weights=[CAPACITY[r] for r in feeders])
        outlet = float(topology["model"]["plant"][plant]["outlet_line"])
        tail = max([outlet] + [levels[r] for r in RECEIVERS[plant]])
        head = np.array([max(level - tail, 1.0)])
        quantities = [float(efficiency(GEN_SPECS[g], head, float(GEN_SPECS[g]["p_max"]))[0][0])
                      for g in PLANT_GENERATORS[plant]]
        qmax[plant] = sum(quantities)
        factor[plant] = sum(float(GEN_SPECS[g]["p_max"]) for g in PLANT_GENERATORS[plant]) * 1e6 / (qmax[plant] * 3600)
    # SHOP describes energy_value_input as local to the first downstream plant.
    # Use a best-point initial-head energy conversion, with main-river flags for
    # river-only reservoirs. The course generation script/settings are not given:
    # this is a documented proxy, not an exact reconstruction of SHOP.
    value_factor = {}
    for plant in s["plants"]:
        feeder = PLANT_FEEDERS[plant]
        level = np.average([levels[r] for r in feeder],weights=[CAPACITY[r] for r in feeder])
        tail = max([float(topology["model"]["plant"][plant]["outlet_line"])] + [levels[r] for r in RECEIVERS[plant]])
        h = np.full(33,max(level-tail,1.))
        powers = [np.linspace(float(GEN_SPECS[g]["p_min"]),float(GEN_SPECS[g]["p_max"]),33) for g in PLANT_GENERATORS[plant]]
        discharge = [efficiency(GEN_SPECS[g],h,power)[0] for g,power in zip(PLANT_GENERATORS[plant],powers)]
        value_factor[plant] = float(np.max(sum(powers)*1e6/(sum(discharge)*3600)))

    def conversion_plants(node,seen=None):
        seen = set() if seen is None else seen
        if node in seen:
            raise ValueError("Cycle in terminal conversion routing.")
        if node[0]=="plant":
            return {node[1]}
        # Preserve the supplied connection-list order for the documented default.
        outgoing = [(edge["to_type"],edge["to"]) for edge in connections
                    if (edge["from_type"],edge["from"])==node and edge["to_type"] in HYDRAULIC_KINDS]
        if node[0]=="reservoir":
            direct = [target for target in outgoing if target[0]=="plant"]
            tunnels = [target for target in outgoing if target[0]=="tunnel"]
            rivers = [target for target in outgoing if target[0]=="river"]
            if direct:
                outgoing = direct
            elif tunnels:
                # Reservoir/tunnel connectivity is the preferred generating
                # route; downstream river bypasses are not extra generation.
                outgoing = tunnels
            elif rivers:
                main = [target for target in rivers if int(topology["model"]["river"][target[1]].get("main_river",0))==1]
                if len(main)>1:
                    raise ValueError("Multiple supplied main rivers at one reservoir.")
                outgoing = main or rivers[:1]
        found = set()
        for target in outgoing:
            found |= conversion_plants(target,seen|{node})
        return found

    selected_plant = {}
    for reservoir in RESERVOIRS:
        candidates = conversion_plants(("reservoir",reservoir))
        if not candidates:
            raise ValueError("No downstream plant for terminal energy conversion.")
        selected_plant[reservoir] = max(candidates,key=lambda plant:(value_factor[plant],plant))
    local_values = dict(zip(RESERVOIRS,terminal))
    global_values = {}
    def global_value(reservoir,seen=None):
        if reservoir in global_values:
            return global_values[reservoir]
        seen = set() if seen is None else seen
        if reservoir in seen:
            raise ValueError("Cycle in terminal water-value approximation.")
        plant = selected_plant[reservoir]
        receiving = max([0.] + [global_value(r,seen|{reservoir}) for r in RECEIVERS[plant]])
        global_values[reservoir] = float(local_values[reservoir])*value_factor[plant]+receiving
        return global_values[reservoir]
    terminal_money = [global_value(r) for r in RESERVOIRS]
    objective = np.zeros(s["nv"])
    rhs = np.zeros(k * nn)
    limits = np.zeros(2 * k * nr)
    bounds = []
    plant_arc = {}
    for step in range(k):
        for edge, (left, right) in enumerate(s["arcs"]):
            low, high = 0.0, 1000.0 * seconds / 1e6
            if left[0] == "plant":
                high = qmax[left[1]] * seconds / 1e6
                objective[step * nf + edge] = -price[step] * factor[left[1]]
                plant_arc[left[1]] = edge
            if left[0] == "river" and left[1] in minflow:
                low = float(minflow[left[1]].to_numpy().reshape(k, -1)[step].sum()) * 3600 / 1e6
            bounds.append((low, max(high, low)))
        for node, position in s["node_position"].items():
            if node[1] in inflow and node[0] in {"reservoir", "river"}:
                rhs[step * nn + position] = float(inflow[node[1]].to_numpy().reshape(k, -1)[step].sum()) * 3600 / 1e6
        for index, reservoir in enumerate(RESERVOIRS):
            if step == 0:
                rhs[s["node_position"][("reservoir", reservoir)]] += volume[index]
            limits[2 * (step * nr + index)] = CAPACITY[reservoir]
            floor = float(minvol[reservoir].to_numpy().reshape(k, -1)[step].max()) if reservoir in minvol else 0.0
            limits[2 * (step * nr + index) + 1] = -floor
    bounds += [(0, None)] * (3 * k * nr)
    for index in range(nr):
        objective[k * nf + (k - 1) * nr + index] = -terminal_money[index]
    objective[k * (nf + nr):] = 1e6
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Unrecognized options detected")
        solved = linprog(objective, A_ub=s["ub"], b_ub=limits, A_eq=s["eq"], b_eq=rhs,
                         bounds=bounds, method="highs", options={"threads": 1})
    if solved.success:
        flow = solved.x[:k * nf].reshape(k, nf)
        storage = solved.x[k * nf:k * (nf + nr)].reshape(k, nr)
        production_fraction = np.column_stack([flow[:, plant_arc[p]] / (qmax[p] * seconds / 1e6) for p in s["plants"]])
        violations = solved.x[k * (nf + nr):].reshape(2, k, nr).sum(axis=(0, 2))
        equality_error = float(np.max(np.abs(s["eq"] @ solved.x - rhs)))
        if equality_error > 1e-5:
            raise ValueError("Allocation water-balance residual is too large.")
    else:
        production_fraction, storage, violations = np.zeros((k, len(s["plants"]))), np.tile(volume, (k, 1)), np.zeros(k)
        equality_error = None
        flow = np.zeros((k, nf))
    if solved.success:
        stationarity = (objective - s["eq"].T @ solved.eqlin.marginals
                        - s["ub"].T @ solved.ineqlin.marginals
                        - solved.lower.marginals - solved.upper.marginals)
        dual_error = float(np.max(np.abs(stationarity)))
        if dual_error > 1e-4:
            raise ValueError("Allocation dual stationarity residual is too large.")
        shadows = solved.eqlin.marginals.reshape(k, nn)
        lower_shadow = solved.lower.marginals[:k*nf].reshape(k, nf)
        upper_shadow = solved.upper.marginals[:k*nf].reshape(k, nf)
    else:
        shadows, lower_shadow, upper_shadow = np.zeros((k, nn)), np.zeros((k, nf)), np.zeros((k, nf))
        dual_error = None
    result = {"production_fraction": production_fraction, "storage": storage,
              "violations": violations, "success": bool(solved.success), "equality_error": equality_error,
              "flows_Mm3": flow, "plant_qmax_m3s": np.array([qmax[p] for p in s["plants"]]),
              "plant_energy_MWh_per_Mm3": np.array([factor[p] for p in s["plants"]]),
              "node_shadow_money_per_Mm3": shadows, "arc_lower_shadow_money_per_Mm3": lower_shadow,
              "arc_upper_shadow_money_per_Mm3": upper_shadow, "dual_stationarity_error": dual_error,
              "objective_money": float(solved.fun) if solved.success else None}
    result["terminal_conversion_plant"] = selected_plant
    result["terminal_bestpoint_energy_MWh_per_Mm3"] = np.array([value_factor[plant] for plant in s["plants"]])
    result["terminal_global_money_per_Mm3"] = np.array(terminal_money)
    ALLOCATION_CACHE[start] = result
    return result


def allocation_features(base, generator):
    starts = base.attrs["case_starts"]
    results = [allocation_case(start) for start in starts]
    structure = ALLOCATION_STRUCTURE
    plant = PLANT_FOR[generator]
    position = structure["plants"].index(plant)
    flows = np.stack([r["production_fraction"] for r in results])
    storage = np.stack([r["storage"] for r in results])
    repeat = lambda value: np.repeat(value, ALLOCATION_STEP_HOURS, axis=1)
    result = {f"allocation_fraction__{p}": repeat(flows[:, :, i]) for i, p in enumerate(structure["plants"])}
    local = repeat(flows[:, :, position])
    result["allocation_local_fraction"] = local
    result["allocation_local_mean_fraction"] = np.repeat(local.mean(1)[:, None], HOURS, axis=1)
    result["allocation_local_remaining_mean_fraction"] = local[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1)
    result["allocation_local_elapsed_mean_fraction"] = local.cumsum(1) / np.arange(1, HOURS + 1)
    result["allocation_local_hours_positive_remaining"] = (local[:, ::-1] > 1e-7).cumsum(1)[:, ::-1]
    for label, reservoirs in [("intake", FEEDERS[generator]), ("upstream", UPSTREAM[generator]),
                              ("receiver", RECEIVERS[plant])]:
        if reservoirs:
            indices = [RESERVOIRS.index(r) for r in reservoirs]
            fraction = storage[:, :, indices].sum(2) / sum(CAPACITY[r] for r in reservoirs)
        else:
            fraction = np.zeros(storage.shape[:2])
        result[f"allocation_projected_{label}_fraction"] = repeat(fraction)
    result["allocation_soft_constraint_violation_Mm3"] = repeat(np.stack([r["violations"] for r in results]))
    result["allocation_failed"] = np.repeat(np.array([not r["success"] for r in results], dtype=float)[:, None], HOURS, axis=1)
    return result

def precompute_allocation_cases(starts):
    """Solve independent case LPs concurrently; all input tables are read-only."""
    from concurrent.futures import ThreadPoolExecutor
    global ALLOCATION_STRUCTURE
    if ALLOCATION_STRUCTURE is None:
        ALLOCATION_STRUCTURE = allocation_structure()
    missing = [start for start in pd.DatetimeIndex(starts).unique() if start not in ALLOCATION_CACHE]
    workers = min(int(BASE_PARAMETERS.get("allocation_precompute_workers",1)),len(missing))
    if workers<=1:
        for start in missing:
            allocation_case(start)
    else:
        with ThreadPoolExecutor(max_workers=workers) as pool:
            list(pool.map(allocation_case,missing))


## Physical, river and operating features

The network receives projected head/efficiency/production, relative economic
coordinates, river geometry and environmental requirements, LP marginal
values and operating windows. Another 54 scenario channels describe
ON/OFF payoff differences under assumed transition costs. These are
continuous physical/economic descriptors, not observed commitment labels.


In [6]:
def projected_hourly_state(base):
    """Interpolate LP block-end storage, never observed post-start storage."""
    starts = base.attrs["case_starts"]
    results = [allocation_case(start) for start in starts]
    initial = np.stack([base[f"initial_volume__{r}"].to_numpy().reshape(-1, HOURS)[:, 0]
                        for r in RESERVOIRS], axis=1)
    block_end = np.stack([r["storage"] for r in results])
    knots = np.concatenate([initial[:, None, :], block_end], axis=1)
    hour = np.arange(HOURS)
    block = hour // ALLOCATION_STEP_HOURS
    fraction = (hour % ALLOCATION_STEP_HOURS)[None, :, None] / ALLOCATION_STEP_HOURS
    volume = knots[:, block] * (1 - fraction) + knots[:, block + 1] * fraction
    levels = {r: np.interp(volume[:, :, j], *LEVEL_CURVES[r]) for j, r in enumerate(RESERVOIRS)}
    return results, volume, levels


def physical_sequence_features(base, groups):
    """Available-input descriptors; loading scenarios are not exact SHOP states."""
    results, volume, levels = projected_hourly_state(base)
    structure = ALLOCATION_STRUCTURE
    n = len(results)
    price = base["price"].to_numpy().reshape(n, HOURS).astype(float)
    price_scale = price.std(1, keepdims=True) + 1.0
    dynamic, static = {}, {}
    def add(name, value):
        value = np.asarray(value, dtype=np.float32)
        if value.shape != (n, HOURS) or not np.isfinite(value).all():
            raise ValueError(f"Invalid physical feature: {name}")
        dynamic[name] = value
    plant_flows = np.stack([r["production_fraction"] for r in results])
    capacity_flows = np.stack([r["plant_qmax_m3s"] for r in results])
    energy_factors = np.stack([r["plant_energy_MWh_per_Mm3"] for r in results])
    cost_per_mw = [(float(GEN_SPECS[g]["startcost_const"]) + float(GEN_SPECS[g]["stopcost_const"]))
                    / float(GEN_SPECS[g]["p_min"]) for g in GENERATORS]
    relative_cost_scale = float(np.median(cost_per_mw))
    for plant, generators in PLANT_GENERATORS.items():
        representative = generators[0]
        spec = GEN_SPECS[representative]
        # Only supplied equal-unit plants are collapsed. Unequal units need their own descriptors.
        compared = ["p_min", "p_max", "turb_eff_curves", "gen_eff_curve", "startcost_const", "stopcost_const"]
        if any(any(GEN_SPECS[g][key] != spec[key] for key in compared) for g in generators):
            raise ValueError("Generator-specific physics requires separate groups for unequal units.")
        index = structure["plants"].index(plant)
        qplant = np.repeat(plant_flows[:, :, index] * capacity_flows[:, None, index], ALLOCATION_STEP_HOURS, axis=1)
        feeder = PLANT_FEEDERS[plant]
        intake = np.min(np.stack([levels[r] for r in feeder]), axis=0)
        tailwater = np.full_like(intake, float(topology["model"]["plant"][plant]["outlet_line"]))
        for r in RECEIVERS[plant]: tailwater = np.maximum(tailwater, levels[r])
        gross = intake - tailwater
        main = np.asarray(topology["model"]["plant"][plant]["main_loss"], dtype=float)
        if len(main) != 1: raise ValueError("Expected one supplied main-tunnel loss coefficient.")
        penstocks = np.asarray(topology["model"]["plant"][plant]["penstock_loss"], dtype=float)
        pipe = int(spec["penstock"]) - 1
        shares = sum(int(GEN_SPECS[g]["penstock"]) == pipe + 1 for g in generators)
        balanced_pipe_flow = qplant * shares / len(generators)
        main_loss = main[0] * qplant**2
        balanced_loss = main_loss + penstocks[pipe] * balanced_pipe_flow**2
        concentrated_pipe = np.minimum(qplant, capacity_flows[:, index, None] * shares / len(generators))
        concentrated_loss = main_loss + penstocks[pipe] * concentrated_pipe**2
        net = np.maximum(gross - balanced_loss, 1.0)
        qmin, eta_min = efficiency(spec, net, float(spec["p_min"]))
        qmax, eta_max = efficiency(spec, net, float(spec["p_max"]))
        factors = 1000 * 9.81 * net * eta_min / 3600
        local_value = sum(base[f"terminal_value__{r}"].to_numpy().reshape(n, HOURS) * CAPACITY[r]
                          for r in feeder) / sum(CAPACITY[r] for r in feeder)
        # Water value is in reference MWh; conversion remains an initial-factor proxy.
        adjusted_margin = price * factors / energy_factors[:, index, None] - local_value
        prefix = f"physical::{plant}::"
        if "hydraulics" in groups:
            add(prefix + "projected_gross_head_m", gross)
            add(prefix + "balanced_net_head_m", net)
            add(prefix + "balanced_loss_fraction", balanced_loss / np.maximum(np.abs(gross), 1))
            add(prefix + "concentrated_extra_loss_fraction", (concentrated_loss-balanced_loss) / np.maximum(np.abs(gross), 1))
            add(prefix + "minimum_point_efficiency", eta_min)
            add(prefix + "maximum_point_efficiency", eta_max)
            add(prefix + "minimum_flow_unit_equivalents", qplant / qmin)
            add(prefix + "maximum_flow_unit_equivalents", qplant / qmax)
            count = np.arange(len(generators)+1)[None, None, :]
            gap = np.maximum(count*qmin[:, :, None]-qplant[:, :, None], 0) + np.maximum(qplant[:, :, None]-count*qmax[:, :, None], 0)
            add(prefix + "distance_to_unit_flow_band", gap.min(2) / capacity_flows[:, index, None])
        if "economics" in groups:
            context = economic_context(adjusted_margin / price_scale, price / price_scale)
            relative_cost = ((float(spec["startcost_const"])+float(spec["stopcost_const"]))
                             / float(spec["p_min"]) / relative_cost_scale)
            add(prefix + "head_adjusted_margin_scaled", adjusted_margin / price_scale)
            add(prefix + "relative_cost_per_opportunity_hour", relative_cost / (1+context["economic_positive_streak_total_h"]))
            add(prefix + "relative_cost_per_margin_area", relative_cost / (1+context["economic_positive_streak_margin_sum"]))
            add(prefix + "head_adjusted_positive_hours_remaining", context["economic_positive_fraction_remaining"])
    if "network" in groups:
        for j, r in enumerate(RESERVOIRS):
            projected = volume[:, :, j] / CAPACITY[r]
            add(f"physical::reservoir::{r}::projected_fraction", projected)
            if f"min_volume__{r}" in base:
                floor = base[f"min_volume__{r}"].to_numpy().reshape(n, HOURS) / CAPACITY[r]
                add(f"physical::reservoir::{r}::margin_to_floor_fraction", projected-floor)
        arc_positions = {edge:i for i,edge in enumerate(structure["arcs"])}
        flows = np.stack([r["flows_Mm3"] for r in results])
        for name, spec in topology["model"]["tunnel"].items():
            upstream = [left for left, rights in FORWARD.items() if ("tunnel", name) in rights]
            outgoing = sorted(FORWARD[("tunnel", name)])
            if len(upstream)!=1 or upstream[0][0]!="reservoir" or len(outgoing)!=1:
                raise ValueError("Unexpected supplied tunnel configuration.")
            source = upstream[0][1]
            target_kind, target_name = outgoing[0]
            q = np.repeat(flows[:, :, arc_positions[(("tunnel",name),outgoing[0])]],ALLOCATION_STEP_HOURS,axis=1)*1e6/(ALLOCATION_STEP_HOURS*3600)
            add(f"physical::tunnel::{name}::entrance_submergence_m", levels[source]-float(spec["start_height"]))
            friction = float(spec["loss_factor"])*q**2
            add(f"physical::tunnel::{name}::modeled_friction_loss_m", friction)
            if target_kind=="reservoir":
                drive = levels[source]-np.maximum(levels[target_name],float(spec["end_height"]))
                add(f"physical::tunnel::{name}::pressure_drive_m", drive)
                add(f"physical::tunnel::{name}::pressure_deficit_proxy_m", friction-drive)
    return dynamic, static


In [7]:
def regime_economic_features(base):
    """Dimensionless within-case economics; no learned or future observed state."""
    n = len(base) // HOURS
    price = base["price"].to_numpy().reshape(n, HOURS).astype(np.float64)
    values = np.column_stack([base[f"terminal_value__{r}"].to_numpy().reshape(n, HOURS)[:, 0]
                              for r in RESERVOIRS]).astype(np.float64)
    # A common positive price/value rescaling cancels unless the numerical floor binds.
    money_scale = np.maximum(np.median(np.abs(np.concatenate([price, values], axis=1)), axis=1), 1.)
    d = {"regime::price_relative": price / money_scale[:, None],
         "regime::price_minus_median_value_relative": (price - np.median(values, axis=1)[:, None]) / money_scale[:, None]}
    s = {"regime::log_money_scale": np.log1p(money_scale)}
    for j, reservoir in enumerate(RESERVOIRS):
        value = values[:, j, None]
        d[f"regime::{reservoir}::margin_relative"] = (price - value) / money_scale[:, None]
        d[f"regime::{reservoir}::symmetric_margin"] = (price - value) / np.maximum(np.abs(price) + np.abs(value), 1.)
        s[f"regime::{reservoir}::terminal_relative"] = values[:, j] / money_scale
    return ({k: np.asarray(v, dtype=np.float32) for k, v in d.items()},
            {k: np.asarray(v, dtype=np.float32) for k, v in s.items()})








def environmental_features(base):
    """Direct river-flow context; residual demand does not uniquely force a plant ON."""
    n = len(base) // HOURS
    constrained = set(tables["min_flow"].columns)
    def river_descendants(node):
        pending, visited = list(FORWARD.get(node, [])), set()
        while pending:
            other = pending.pop()
            if other in visited or other[0] != "river":
                continue
            visited.add(other)
            pending.extend(FORWARD.get(other, []))
        return {name for kind, name in visited}
    # Only river-to-river paths credit minimum-flow support. Reservoirs may store
    # water, so required flow upstream of a reservoir is not instant downstream supply.
    support = {r: [q for q in constrained if q != r and r in river_descendants(("river", q))]
               for r in constrained}
    dynamic, static = {}, {}
    results = None
    for plant, generators in PLANT_GENERATORS.items():
        direct = sorted(constrained & river_descendants(("plant", plant)))
        if not direct:
            continue
        if results is None:
            results = [allocation_case(start) for start in base.attrs["case_starts"]]
        deficit = np.zeros((n, HOURS), dtype=float)
        required = np.zeros_like(deficit)
        for river in direct:
            minimum = base[f"min_flow__{river}"].to_numpy().reshape(n, HOURS)
            natural = (base[f"inflow__{river}"].to_numpy().reshape(n, HOURS)
                       if f"inflow__{river}" in base else np.zeros_like(minimum))
            known_support = sum((base[f"min_flow__{q}"].to_numpy().reshape(n, HOURS) for q in support[river]),
                                np.zeros_like(minimum))
            deficit = np.maximum(deficit, np.maximum(minimum-natural-known_support, 0))
            required = np.maximum(required, minimum)
        levels = {r: np.interp(base[f"initial_volume__{r}"].to_numpy().reshape(n, HOURS)[:, 0], *LEVEL_CURVES[r]) for r in RESERVOIRS}
        source = np.average(np.stack([levels[r] for r in PLANT_FEEDERS[plant]]), axis=0,
                            weights=[CAPACITY[r] for r in PLANT_FEEDERS[plant]])
        outlet = np.maximum.reduce([np.full(n, float(topology["model"]["plant"][plant]["outlet_line"]))]
                                    + [levels[r] for r in RECEIVERS[plant]])
        head = np.maximum(source-outlet, 1.)
        qmin = sum(efficiency(GEN_SPECS[g], head, float(GEN_SPECS[g]["p_min"]))[0] for g in generators)
        index = ALLOCATION_STRUCTURE["plants"].index(plant)
        qmax = np.asarray([r["plant_qmax_m3s"][index] for r in results])
        modeled = np.repeat(np.stack([r["production_fraction"][:, index] for r in results]), ALLOCATION_STEP_HOURS, axis=1)*qmax[:, None]
        price = base["price"].to_numpy().reshape(n, HOURS)
        value = sum(base[f"terminal_value__{r}"].to_numpy().reshape(n, HOURS)*CAPACITY[r] for r in PLANT_FEEDERS[plant])/sum(CAPACITY[r] for r in PLANT_FEEDERS[plant])
        money_scale = np.maximum(np.median(np.abs(price), axis=1), 1.)[:, None]
        fraction = deficit/np.maximum(required, 1e-6)
        prefix = f"environment::{plant}::"
        dynamic[prefix+"residual_fraction_of_required_flow"] = fraction
        dynamic[prefix+"residual_per_minimum_plant_flow"] = deficit/qmin[:, None]
        dynamic[prefix+"residual_per_maximum_plant_flow"] = deficit/qmax[:, None]
        dynamic[prefix+"modeled_flow_relative_to_residual"] = modeled/(deficit+qmin[:, None])
        dynamic[prefix+"margin_under_environmental_deficit"] = ((price-value)/money_scale)*fraction
        if BASE_PARAMETERS.get("bypass_scenario_features", False):
            # Conditional alternatives: release the residual by bypass, or generate
            # at the operating point. Only extra water above that release is
            # charged its local terminal opportunity value. Other supply routes,
            # receiving-water values and binary startup effects remain approximate.
            minimum_extra_fraction = np.clip(1-deficit/qmin[:, None], 0, 1)
            maximum_extra_fraction = np.clip(1-deficit/qmax[:, None], 0, 1)
            minimum_margin = (price-value*minimum_extra_fraction)/money_scale
            dynamic[prefix+"minimum_loading_bypass_replacement_margin_relative"] = minimum_margin
            dynamic[prefix+"maximum_loading_bypass_replacement_margin_relative"] = (price-value*maximum_extra_fraction)/money_scale
            positive = minimum_margin > 0
            dynamic[prefix+"minimum_loading_positive_fraction_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1]/np.arange(HOURS, 0, -1)
        static[prefix+"week_fraction_with_residual"] = (deficit>1e-6).mean(1)
    return ({k:np.asarray(v,dtype=np.float32) for k,v in dynamic.items()},
            {k:np.asarray(v,dtype=np.float32) for k,v in static.items()})


def allocation_dual_features(base):
    """Normalized sensitivities of the same coarse continuous relaxation, not SHOP duals."""
    n = len(base) // HOURS
    results = [allocation_case(start) for start in base.attrs["case_starts"]]
    s = ALLOCATION_STRUCTURE
    shadow = np.stack([r["node_shadow_money_per_Mm3"] for r in results])
    lower = np.stack([r["arc_lower_shadow_money_per_Mm3"] for r in results])
    upper = np.stack([r["arc_upper_shadow_money_per_Mm3"] for r in results])
    energy = np.stack([r["plant_energy_MWh_per_Mm3"] for r in results])
    price = base["price"].to_numpy().reshape(n, HOURS)
    scale = np.maximum(np.median(np.abs(price), axis=1), 1.)[:, None]
    repeat = lambda value: np.repeat(value, ALLOCATION_STEP_HOURS, axis=1)
    out = {}
    for j, plant in enumerate(s["plants"]):
        positions = [i for i, (left, right) in enumerate(s["arcs"]) if left == ("plant", plant)]
        if len(positions) != 1:
            raise ValueError("Dual plant descriptors require a single discharge arc.")
        edge = positions[0]
        left, right = s["arcs"][edge]
        left_value = shadow[:, :, s["node_position"][left]]
        right_value = shadow[:, :, s["node_position"][right]] if right in s["node_position"] else np.zeros_like(left_value)
        # SciPy marginals are objective derivatives: c - A.T@y equals bound marginals.
        # The negative production-reward sign makes (right-left)/energy the
        # effective network water cost in reference MWh units at the plant arc.
        water_cost = repeat((right_value-left_value)/energy[:, j, None])
        prefix = f"dual::{plant}::"
        out[prefix+"hourly_margin_relative"] = (price-water_cost)/scale
        out[prefix+"network_water_cost_relative"] = water_cost/scale
        # Off-plant duals can place the reduced cost on an intake arc instead of
        # the discharge arc. Expose the cheapest feeder reservoir sensitivity too.
        feeder_shadow = np.maximum.reduce([shadow[:, :, s["node_position"][("reservoir",r)]]
                                           for r in PLANT_FEEDERS[plant]])
        reservoir_cost = repeat((right_value-feeder_shadow)/energy[:, j, None])
        out[prefix+"feeder_water_cost_relative"] = reservoir_cost/scale
        out[prefix+"feeder_hourly_margin_relative"] = (price-reservoir_cost)/scale
        out[prefix+"capacity_scarcity_relative"] = repeat(-upper[:, :, edge]/energy[:, j, None])/scale
        out[prefix+"lower_bound_cost_relative"] = repeat(lower[:, :, edge]/energy[:, j, None])/scale
    factor = np.median(energy, axis=1)[:, None]
    for river in tables["min_flow"].columns:
        edge = next(i for i,(left,right) in enumerate(s["arcs"]) if left == ("river",river))
        out[f"dual::river::{river}::minimum_flow_cost_relative"] = repeat(lower[:, :, edge])/factor/scale
    return {k:np.asarray(v,dtype=np.float32) for k,v in out.items()}




def network_economic_context(base):
    """Within-case opportunity duration relative to the relaxed network water cost."""
    n = len(base) // HOURS
    duals = allocation_dual_features(base)
    price = base["price"].to_numpy().reshape(n, HOURS).astype(np.float64)
    scale = np.maximum(np.median(np.abs(price), axis=1), 1.)[:, None]
    selected = ("economic_positive_streak_total_h", "economic_positive_streak_margin_sum",
                "economic_hours_to_next_positive", "economic_positive_fraction_remaining",
                "economic_next_6h_margin_mean", "economic_next_24h_margin_mean")
    out = {}
    for plant in PLANT_GENERATORS:
        margin = duals[f"dual::{plant}::feeder_hourly_margin_relative"]
        # Ignore floating-point noise at an LP break-even point, not real EUR thresholds.
        margin = np.where(np.abs(margin)<1e-6, 0., margin)
        context = economic_context(margin, price/scale)
        for name in selected:
            out[f"dual_context::{plant}::{name}"] = np.asarray(context[name], dtype=np.float32)
    return out


def river_penalty_proxy(flow, specification):
    """Integral of supplied nonnegative rate segments; descriptor, not SHOP output."""
    q = np.maximum(np.asarray(flow, dtype=float), 0.)
    if "flow_cost_curve" not in specification:
        return q * float(specification.get("flow_cost_const", 0.))
    curve = specification["flow_cost_curve"]
    x, y = np.asarray(curve["x"], dtype=float), np.asarray(curve["y"], dtype=float)
    if len(x) != len(y) or x[0] != 0 or np.any(np.diff(x) <= 0) or np.any(y < 0):
        raise ValueError("Unexpected supplied river cost-rate curve.")
    value = np.zeros_like(q)
    for j in range(len(x)-1):
        value += np.clip(q-x[j], 0, x[j+1]-x[j]) * y[j]
    # The final documented cost rate also covers flow beyond the last breakpoint.
    return value + np.maximum(q-x[-1], 0) * y[-1]


def river_geometry_features(base):
    """River geometry and within-currency route-cost descriptors from allowed inputs."""
    results, _, levels = projected_hourly_state(base)
    structure = ALLOCATION_STRUCTURE
    flows = np.stack([r["flows_Mm3"] for r in results])
    dynamic, static, costs = {}, {}, {}
    hourly_flow = {}
    references = []
    for river, spec in topology["model"]["river"].items():
        edges = [i for i,(left,right) in enumerate(structure["arcs"]) if left == ("river",river)]
        if len(edges) != 1:
            raise ValueError("River descriptors require one discharge arc per supplied river.")
        q = np.repeat(flows[:, :, edges[0]], ALLOCATION_STEP_HOURS, axis=1)*1e6/(ALLOCATION_STEP_HOURS*3600)
        q = np.maximum(q, 0.)
        hourly_flow[river] = q
        if "flow_cost_curve" in spec:
            references.extend(float(y) for y in spec["flow_cost_curve"]["y"] if float(y)>0)
        elif float(spec.get("flow_cost_const",0))>0:
            references.append(float(spec["flow_cost_const"]))
        if "flow_cost_curve" in spec or float(spec.get("flow_cost_const",0))>0:
            costs[river] = river_penalty_proxy(q, spec)
        curves = spec.get("up_head_flow_curve")
        if curves:
            parents = [left for left,rights in FORWARD.items() if ("river",river) in rights]
            if len(parents)!=1 or parents[0][0]!="reservoir" or len(curves)!=1:
                raise ValueError("Unexpected supplied level-dependent river geometry.")
            reservoir = parents[0][1]
            curve = curves[0]
            x,y = np.asarray(curve["x"],dtype=float), np.asarray(curve["y"],dtype=float)
            if np.any(np.diff(x)<=0) or np.any(np.diff(y)<=0):
                raise ValueError("The supplied head/discharge curve must be monotone.")
            level = levels[reservoir]
            # Bounded curve coordinates and a separate extrapolation indicator avoid
            # pretending to solve unprovided gate/MIP settings or exact spill physics.
            required = np.interp(q, y, x)
            span = max(float(topology["model"]["reservoir"][reservoir]["hrl"])-float(topology["model"]["reservoir"][reservoir]["lrl"]),1.)
            prefix=f"river::{river}::"
            dynamic[prefix+"crest_clearance_fraction"] = (level-float(spec["upstream_elevation"]))/span
            dynamic[prefix+"curve_head_gap_fraction"] = (level-required)/span
            dynamic[prefix+"curve_flow_range_fraction"] = q/max(float(y[-1]),1.)
            dynamic[prefix+"above_curve_range_fraction"] = np.maximum(q-float(y[-1]),0)/max(float(y[-1]),1.)
    if not costs or not references:
        raise ValueError("No supplied river cost descriptors were found.")
    # All river rates use the same monetary unit. Normalizing by another supplied
    # rate preserves relative route penalties without importing an exchange rate.
    rate_reference = float(np.median(references))
    total_flow = sum(hourly_flow.values())
    flow_reference = np.maximum(total_flow.mean(1,keepdims=True),1e-6)
    total_cost = sum(costs.values())
    week_cost = total_cost.sum(1)
    for river,cost in costs.items():
        prefix=f"river::{river}::"
        dynamic[prefix+"normalized_penalty_rate"] = cost/(rate_reference*flow_reference)
        dynamic[prefix+"penalty_share"] = np.divide(cost,total_cost,out=np.zeros_like(cost),where=total_cost>0)
        static[prefix+"week_penalty_share"] = np.divide(cost.sum(1),week_cost,out=np.zeros_like(week_cost),where=week_cost>0)
    dynamic["river::system::relative_route_cost"] = np.divide(total_cost,rate_reference*total_flow,out=np.zeros_like(total_cost),where=total_flow>0)
    output_d = {k:np.asarray(v,dtype=np.float32) for k,v in dynamic.items()}
    output_s = {k:np.asarray(v,dtype=np.float32) for k,v in static.items()}
    if not all(np.isfinite(v).all() for v in [*output_d.values(),*output_s.values()]):
        raise ValueError("Nonfinite river descriptor.")
    return output_d,output_s


def binary_commitment_gap(gain, start_cost, stop_cost, initial_free):
    """Exact two-state maximum-payoff difference conditional on each hour ON/OFF.

    Payoffs/costs are normalized opportunity scenarios, not an asserted EUR/NOK
    conversion. Initial free and initial OFF are separate input scenarios.
    """
    gain = np.asarray(gain,dtype=float)
    if gain.ndim!=2 or gain.shape[1]<1 or not np.isfinite(gain).all() or min(start_cost,stop_cost)<0:
        raise ValueError("Invalid binary opportunity scenario.")
    n,width=gain.shape
    forward_off=np.zeros_like(gain);forward_on=np.zeros_like(gain)
    off=np.zeros(n);on=np.zeros(n) if initial_free else np.full(n,-np.inf)
    for h in range(width):
        next_off=np.maximum(off,on-stop_cost)
        next_on=gain[:,h]+np.maximum(on,off-start_cost)
        forward_off[:,h],forward_on[:,h]=next_off,next_on
        off,on=next_off,next_on
    backward_off=np.zeros_like(gain);backward_on=np.zeros_like(gain)
    for h in range(width-2,-1,-1):
        backward_off[:,h]=np.maximum(backward_off[:,h+1],gain[:,h+1]-start_cost+backward_on[:,h+1])
        backward_on[:,h]=np.maximum(gain[:,h+1]+backward_on[:,h+1],-stop_cost+backward_off[:,h+1])
    return forward_on+backward_on-forward_off-backward_off


def commitment_sequence_features(base):
    """Binary transition-cost scenarios on known weekly network opportunities.

    Water and loading are not reoptimized under commitment. Relative costs per
    rated MW and a predeclared scale grid avoid inventing an exchange rate.
    """
    duals=allocation_dual_features(base)
    unit_cost=[(float(GEN_SPECS[g]["startcost_const"])+float(GEN_SPECS[g]["stopcost_const"]))/float(GEN_SPECS[g]["p_max"]) for g in GENERATORS]
    typical=max(float(np.median(unit_cost)),1e-8)
    out={}
    for plant,generators in PLANT_GENERATORS.items():
        spec=GEN_SPECS[generators[0]]
        start=float(spec["startcost_const"]);stop=float(spec["stopcost_const"])
        relative=(start+stop)/float(spec["p_max"])/typical
        fraction=start/max(start+stop,1e-8)
        gain=duals[f"dual::{plant}::feeder_hourly_margin_relative"]
        gain=np.where(np.abs(gain)<1e-6,0.,gain)
        for scale in (.1,1.,10.):
            penalty=scale*relative
            for initial_free in (False,True):
                label="free" if initial_free else "off"
                name=f"commitment_dp::{plant}::scale_{scale:g}::initial_{label}_gap"
                out[name]=binary_commitment_gap(gain,penalty*fraction,penalty*(1-fraction),initial_free).astype(np.float32)
    return out


## Assemble the hourly and static input tensors

Each input retains its original position. Hourly inputs vary across the week,
while static inputs describe the case as a whole. Every generator's output
column is mapped by name to the supplied template.


In [8]:
def prediction_metadata(cases):
    return cases.drop(columns=LABELS + ["total_calculation_time"], errors="ignore").copy()

def assemble_predictions(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("Predictions must cover all 14 generators.")
    values = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for generator in GENERATORS:
        probability = np.asarray(predictions[generator])
        if probability.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {generator}")
        values[:, TARGET_POSITIONS[generator]] = probability
    if not np.isfinite(values).all() or (values < 0).any() or (values > 1).any():
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    result = pd.DataFrame(values, columns=LABELS)
    result.insert(0, "Run No", cases["Run No"].to_numpy())
    return result

INTAKE_GROUPS = list(dict.fromkeys(tuple(FEEDERS[g]) for g in GENERATORS))
GROUP_REPRESENTATIVES = [next(g for g in GENERATORS if tuple(FEEDERS[g]) == intake)
                         for intake in INTAKE_GROUPS]
STATIC_PREFIXES = ("initial_volume__", "initial_fraction__", "terminal_value__",
                   "inflow_week_mean__", "min_flow_week_max__", "min_volume_week_max__",
                   "price_week_mean", "price_week_std", "price_week_min", "price_week_max")
LOCAL_STATIC = {"local_initial_fraction", "local_terminal_value_capacity_weighted",
                "local_available_water_proxy_fraction", "water_end_no_release_excess_fraction"}

def sequence_inputs(cases):
    precompute_allocation_cases(cases.starttime)
    base = feature_matrix(cases)
    n = len(cases)
    base.attrs["case_starts"] = pd.DatetimeIndex(cases.starttime)
    dynamic_names = [name for name in base if not name.startswith(STATIC_PREFIXES)]
    static_names = [name for name in base if name.startswith(STATIC_PREFIXES)]
    dynamic = [base[dynamic_names].to_numpy().reshape(n, HOURS, -1)]
    static = [base[static_names].to_numpy().reshape(n, HOURS, -1)[:, 0]]
    # Repeated calendar coordinates are more naturally encoded on a circle for a NN.
    h = np.arange(HOURS)[None, :]
    times = pd.DatetimeIndex(np.repeat(cases.starttime.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    cycles = np.stack([np.sin(2*np.pi*times.hour/24), np.cos(2*np.pi*times.hour/24),
                       np.sin(2*np.pi*times.dayofweek/7), np.cos(2*np.pi*times.dayofweek/7)], axis=1)
    dynamic.append(cycles.reshape(n, HOURS, 4).astype(np.float32))
    dynamic_names += ["hour_sin", "hour_cos", "weekday_sin", "weekday_cos"]
    for group_index, generator in enumerate(GROUP_REPRESENTATIVES):
        extended = generator_features(base, generator)
        local = [name for name in extended if name not in base]
        local_dynamic = [name for name in local if name not in LOCAL_STATIC]
        local_static = [name for name in local if name in LOCAL_STATIC]
        dynamic.append(extended[local_dynamic].to_numpy().reshape(n, HOURS, -1))
        static.append(extended[local_static].to_numpy().reshape(n, HOURS, -1)[:, 0])
        dynamic_names += [f"intake_{group_index}::{name}" for name in local_dynamic]
        static_names += [f"intake_{group_index}::{name}" for name in local_static]
        del extended
    if FEATURE_SET.startswith("allocation"):
        # Global plant flows occur once; local summaries retain their intake identity.
        for i, generator in enumerate(GROUP_REPRESENTATIVES):
            values = allocation_features(base, generator)
            if i == 0:
                common = [name for name in values if name.startswith("allocation_fraction__")]
                common += ["allocation_soft_constraint_violation_Mm3", "allocation_failed"]
                dynamic.append(np.stack([values[name] for name in common], axis=2))
                dynamic_names += common
            local = [name for name in values if name.startswith("allocation_local_") or
                     name.startswith("allocation_projected_")]
            local.remove("allocation_local_fraction")  # Already a named global plant flow.
            static_name = "allocation_local_mean_fraction"
            static.append(values[static_name][:, :1])
            static_names.append(f"allocation::intake_{i}::{static_name}")
            local.remove(static_name)
            dynamic.append(np.stack([values[name] for name in local], axis=2))
            dynamic_names += [f"allocation::intake_{i}::{name}" for name in local]
    if PHYSICS_GROUPS:
        physical_dynamic, physical_static = physical_sequence_features(base, PHYSICS_GROUPS)
        if physical_dynamic:
            dynamic.append(np.stack(list(physical_dynamic.values()), axis=2))
            dynamic_names += list(physical_dynamic)
        if physical_static:
            static.append(np.column_stack(list(physical_static.values())))
            static_names += list(physical_static)
    if BASE_PARAMETERS.get("regime_features", False):
        extra_d, extra_s = regime_economic_features(base)
        dynamic.append(np.stack(list(extra_d.values()), axis=2))
        dynamic_names += list(extra_d)
        static.append(np.column_stack(list(extra_s.values())))
        static_names += list(extra_s)
    if BASE_PARAMETERS.get("environment_features", False):
        extra_d, extra_s = environmental_features(base)
        if extra_d:
            dynamic.append(np.stack(list(extra_d.values()), axis=2))
            dynamic_names += list(extra_d)
        if extra_s:
            static.append(np.column_stack(list(extra_s.values())))
            static_names += list(extra_s)
    if BASE_PARAMETERS.get("dual_features", False):
        extra_d = allocation_dual_features(base)
        dynamic.append(np.stack(list(extra_d.values()), axis=2))
        dynamic_names += list(extra_d)
    if BASE_PARAMETERS.get("dual_context_features", False):
        extra_d = network_economic_context(base)
        dynamic.append(np.stack(list(extra_d.values()), axis=2))
        dynamic_names += list(extra_d)
    if BASE_PARAMETERS.get("river_features", False):
        extra_d,extra_s = river_geometry_features(base)
        dynamic.append(np.stack(list(extra_d.values()),axis=2))
        dynamic_names += list(extra_d)
        static.append(np.column_stack(list(extra_s.values())))
        static_names += list(extra_s)
    if BASE_PARAMETERS.get("commitment_dp_features",False):
        extra=commitment_sequence_features(base)
        dynamic.append(np.stack(list(extra.values()),axis=2))
        dynamic_names+=list(extra)
    dynamic = np.ascontiguousarray(np.concatenate(dynamic, axis=2), dtype=np.float32)
    static = np.ascontiguousarray(np.concatenate(static, axis=1), dtype=np.float32)
    if not np.isfinite(dynamic).all() or not np.isfinite(static).all():
        raise ValueError("Sequence inputs must be finite.")
    return dynamic, static, dynamic_names, static_names

def sequence_labels(cases):
    return np.stack([cases[TARGET_COLUMNS[g]].to_numpy(dtype=np.float32)
                     for g in GENERATORS], axis=2)

def sequence_frame(cases, probabilities):
    values = np.asarray(probabilities)
    if values.shape != (len(cases), HOURS, len(GENERATORS)):
        raise ValueError("Expected (cases, 168, 14) probabilities.")
    return assemble_predictions(cases, {g: values[:, :, j] for j, g in enumerate(GENERATORS)})


probe_cases = pd.concat([train_cases.iloc[[0, 21]], test_cases.iloc[[0, -1]]], ignore_index=True)
probe_dynamic, probe_static, DYNAMIC_NAMES, STATIC_NAMES = sequence_inputs(probe_cases)
if (len(DYNAMIC_NAMES), len(STATIC_NAMES)) != (913, 182):
    raise ValueError("Unexpected TCN input schema.")
print(f"Inputs: {len(DYNAMIC_NAMES)} hourly channels and {len(STATIC_NAMES)} static channels")
del probe_dynamic, probe_static


Inputs: 913 hourly channels and 182 static channels


## Scaling and temporal network

Scaling statistics are fitted only on the labeled fitting cases. Asinh
compresses extreme magnitudes while preserving their signs, followed by
standardization and clipping to [-10, 10]. The width-64 backbone has kernel-3
convolutions and dilations 1, 2, 4, 8, 16, 32 and 64. Symmetric padding is
appropriate here because the entire exogenous planning week is supplied.


In [9]:
def fit_scaler(dynamic, static, clip):
    encoding = BASE_PARAMETERS.get("input_encoding", "standard")
    if encoding not in {"standard", "asinh"}:
        raise ValueError("Unknown input encoding.")
    result = {"clip": float(clip), "encoding": np.asarray(encoding)}
    for name, raw in (("dynamic", dynamic), ("static", static)):
        if encoding == "asinh":
            # Deterministic subset of fitting cases only; bound temporary sort memory.
            rows = np.linspace(0, len(raw)-1, min(len(raw), 256), dtype=int)
            sample = raw[rows, ::7] if raw.ndim == 3 else raw[rows]
            sample = sample.reshape(-1, raw.shape[-1])
            scale = np.maximum(np.quantile(np.abs(sample), .75, axis=0), 1e-3).astype(np.float32)
            value = raw.copy()
            value /= scale
            np.arcsinh(value, out=value)
        else:
            scale = np.ones(raw.shape[-1], dtype=np.float32)
            value = raw
        axes = tuple(range(raw.ndim-1))
        result[name+"_transform_scale"] = scale
        result[name+"_mean"] = value.mean(axis=axes, dtype=np.float64).astype(np.float32)
        result[name+"_std"] = np.maximum(value.std(axis=axes, dtype=np.float64), 1e-6).astype(np.float32)
        del value
    return result


def scale_inputs(dynamic, static, scaler):
    output = []
    encoding = str(scaler.get("encoding", "standard"))
    for name, raw in (("dynamic", dynamic), ("static", static)):
        value = np.ascontiguousarray(raw, dtype=np.float32).copy()
        if encoding == "asinh":
            value /= scaler[name+"_transform_scale"]
            np.arcsinh(value, out=value)
        value -= scaler[name+"_mean"]
        value /= scaler[name+"_std"]
        np.clip(value, -float(scaler["clip"]), float(scaler["clip"]), out=value)
        output.append(value)
    return tuple(output)


def initialize_torch():
    import torch
    torch.set_num_threads(THREADS)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.use_deterministic_algorithms(True)
    return torch


def make_model(parameters):
    import torch
    from torch import nn
    class ChannelNorm(nn.Module):
        def __init__(self, width):
            super().__init__()
            self.norm = nn.LayerNorm(width)
        def forward(self, x):
            return self.norm(x.transpose(1, 2)).transpose(1, 2)
    class ResidualBlock(nn.Module):
        def __init__(self, width, dilation):
            super().__init__()
            layers = []
            for _ in range(2):
                layers += [nn.Conv1d(width, width, parameters["kernel_size"],
                                    dilation=dilation, padding=dilation),
                           ChannelNorm(width), nn.GELU(), nn.Dropout(parameters["dropout"])]
            self.layers = nn.Sequential(*layers)
        def forward(self, x):
            return x + self.layers(x)
    class LocalOpportunityHead(nn.Module):
        """Joint nonlinear skip corrections from named local or global inputs."""
        def __init__(self, hidden, mode):
            super().__init__()
            groups = []
            if mode == "global":
                groups = [(list(range(len(DYNAMIC_NAMES))), list(range(len(STATIC_NAMES))), list(range(len(GENERATORS))))]
            elif mode == "plant":
                common = {"price", "price_minus_week_mean", "price_week_rank", "horizon_hour", "hours_remaining", "hour_sin", "hour_cos", "weekday_sin", "weekday_cos"}
                for i, representative in enumerate(GROUP_REPRESENTATIVES):
                    plant = PLANT_FOR[representative]
                    prefixes = (f"intake_{i}::", f"allocation::intake_{i}::", f"physical::{plant}::")
                    local_d = [j for j, name in enumerate(DYNAMIC_NAMES) if name in common or name.startswith(prefixes) or f"::{plant}::" in name]
                    initial = {f"{prefix}__{reservoir}" for prefix in ("initial_volume", "terminal_value") for reservoir in FEEDERS[representative]}
                    local_s = [j for j, name in enumerate(STATIC_NAMES) if name.startswith(prefixes) or f"::{plant}::" in name or name in initial]
                    if parameters.get("local_hydraulic_routing", False):
                        # Route only named adjacent hydraulic context, not every global reservoir.
                        related_reservoirs = set(FEEDERS[representative]) | set(RECEIVERS[plant])
                        related_nodes = {("reservoir", r) for r in related_reservoirs}
                        related_tunnels = {name for name in topology["model"]["tunnel"]
                            if any(("tunnel", name) in FORWARD.get(node, []) for node in related_nodes)
                            or any(node in FORWARD.get(("tunnel", name), []) for node in related_nodes)}
                        related_rivers = {name for name in topology["model"]["river"]
                            if any(("river", name) in FORWARD.get(node, []) for node in related_nodes)
                            or any(node in FORWARD.get(("river", name), []) for node in related_nodes)}
                        extra_prefixes = tuple([f"physical::reservoir::{r}::" for r in sorted(related_reservoirs)]
                            + [f"regime::{r}::" for r in sorted(related_reservoirs)]
                            + [f"physical::tunnel::{name}::" for name in sorted(related_tunnels)]
                            + [f"river::{name}::" for name in sorted(related_rivers)]
                            + [f"dual::river::{name}::" for name in sorted(related_rivers)])
                        local_d = sorted(set(local_d) | {j for j, name in enumerate(DYNAMIC_NAMES) if name.startswith(extra_prefixes)})
                        local_s = sorted(set(local_s) | {j for j, name in enumerate(STATIC_NAMES) if name.startswith(extra_prefixes)})
                    outputs = [j for j, generator in enumerate(GENERATORS) if PLANT_FOR[generator] == plant]
                    if not local_d or not local_s or not outputs:
                        raise ValueError("Plant input grouping is incomplete.")
                    groups.append((local_d, local_s, outputs))
            else:
                raise ValueError("Unknown nonlinear input mode.")
            covered = [j for _, _, outputs in groups for j in outputs]
            if sorted(covered) != list(range(len(GENERATORS))):
                raise ValueError("Local output groups must cover each generator exactly once.")
            self.layers = nn.ModuleList()
            self.group_count = len(groups)
            for i, (dynamic_indices, static_indices, outputs) in enumerate(groups):
                self.register_buffer(f"dynamic_{i}", torch.tensor(dynamic_indices, dtype=torch.long), persistent=False)
                self.register_buffer(f"static_{i}", torch.tensor(static_indices, dtype=torch.long), persistent=False)
                self.register_buffer(f"outputs_{i}", torch.tensor(outputs, dtype=torch.long), persistent=False)
                layer = nn.Sequential(nn.Linear(len(dynamic_indices)+len(static_indices), hidden),
                                      nn.LayerNorm(hidden), nn.GELU(), nn.Linear(hidden, len(outputs)))
                nn.init.zeros_(layer[-1].weight)
                nn.init.zeros_(layer[-1].bias)
                self.layers.append(layer)
        def forward(self, dynamic, static):
            result = dynamic.new_zeros(dynamic.shape[0], dynamic.shape[1], len(GENERATORS))
            for i, layer in enumerate(self.layers):
                selected_d = dynamic.index_select(2, getattr(self, f"dynamic_{i}"))
                selected_s = static.index_select(1, getattr(self, f"static_{i}")).unsqueeze(1).expand(-1, dynamic.shape[1], -1)
                correction = layer(torch.cat([selected_d, selected_s], dim=2))
                result = result.index_copy(2, getattr(self, f"outputs_{i}"), correction)
            return result
    class TypedFeatureEncoder(nn.Module):
        """Joint residual nonlinear encoders for distinct input mechanisms."""
        def __init__(self, hidden, width):
            super().__init__()
            def kind(name):
                prefix = name.split("::",1)[0]
                if prefix == "physical": return "physics"
                if prefix == "river": return "rivers"
                if prefix in {"commitment_dp","dual_context"}: return "persistence"
                if prefix.startswith("allocation") or prefix == "dual": return "allocation"
                if prefix.startswith("intake_"):
                    return "economics" if any(word in name for word in ["economic", "price", "terminal_value"]) else "water"
                if prefix == "regime" or name.startswith(("price","terminal_value")): return "economics"
                if prefix == "environment" or name.startswith(("initial_","inflow","min_flow","min_volume")): return "water"
                if name in {"horizon_hour","hours_remaining","hour_of_day","day_of_week","month","day_of_year_sin","day_of_year_cos","hour_sin","hour_cos","weekday_sin","weekday_cos"}: return "calendar"
                raise ValueError(f"Unclassified typed input: {name}")
            names = sorted(set(map(kind,DYNAMIC_NAMES+STATIC_NAMES)))
            self.encoders = nn.ModuleList()
            self.group_count = len(names)
            seen_d, seen_s = [], []
            for i, label in enumerate(names):
                di = [j for j,name in enumerate(DYNAMIC_NAMES) if kind(name)==label]
                si = [j for j,name in enumerate(STATIC_NAMES) if kind(name)==label]
                seen_d += di;seen_s += si
                self.register_buffer(f"dynamic_{i}",torch.tensor(di,dtype=torch.long),persistent=False)
                self.register_buffer(f"static_{i}",torch.tensor(si,dtype=torch.long),persistent=False)
                module = nn.Sequential(nn.Linear(len(di)+len(si),hidden),nn.LayerNorm(hidden),nn.GELU(),nn.Linear(hidden,width))
                nn.init.zeros_(module[-1].weight);nn.init.zeros_(module[-1].bias)
                self.encoders.append(module)
            if sorted(seen_d)!=list(range(len(DYNAMIC_NAMES))) or sorted(seen_s)!=list(range(len(STATIC_NAMES))):
                raise ValueError("Typed encoders must cover each model input exactly once.")
        def forward(self,dynamic,static):
            result = dynamic.new_zeros(dynamic.shape[0],dynamic.shape[1],parameters["width"])
            for i,module in enumerate(self.encoders):
                d=dynamic.index_select(2,getattr(self,f"dynamic_{i}"))
                s=static.index_select(1,getattr(self,f"static_{i}")).unsqueeze(1).expand(-1,dynamic.shape[1],-1)
                result=result+module(torch.cat([d,s],dim=2))
            return (result / np.sqrt(self.group_count)).transpose(1,2)
    class HydroTCN(nn.Module):
        def __init__(self, outputs):
            super().__init__()
            width = parameters["width"]
            self.projection = nn.Conv1d(len(DYNAMIC_NAMES), width, 1)
            self.condition = nn.Sequential(nn.Linear(len(STATIC_NAMES) + len(DYNAMIC_NAMES), width),
                                           nn.GELU(), nn.Linear(width, width))
            self.blocks = nn.Sequential(*[ResidualBlock(width, dilation) for dilation in parameters["dilations"]])
            if parameters.get("head_width", 0):
                head_width = int(parameters["head_width"])
                # Disjoint grouped temporal heads specialize each output after a shared encoder.
                self.head = nn.Sequential(ChannelNorm(width), nn.GELU(),
                    nn.Conv1d(width, outputs*head_width, 1), nn.GELU(),
                    nn.Conv1d(outputs*head_width, outputs*head_width, 3, padding=1, groups=outputs),
                    nn.GELU(), nn.Conv1d(outputs*head_width, outputs, 1, groups=outputs))
            else:
                self.head = nn.Sequential(ChannelNorm(width), nn.GELU(), nn.Conv1d(width, outputs, 1))
            self.linear_skip = nn.Conv1d(len(DYNAMIC_NAMES), outputs, 1)
            self.static_skip = nn.Linear(len(STATIC_NAMES), outputs)
            self.local_head = None
            if parameters.get("local_head_width", 0):
                if parameters.get("layout", "shared") != "shared":
                    raise ValueError("The nonlinear input correction requires one shared network.")
                # Preserve the baseline initialization and dropout random stream.
                with torch.random.fork_rng(devices=[]):
                    self.local_head = LocalOpportunityHead(int(parameters["local_head_width"]), parameters["local_input_mode"])
            self.typed_input = None
            if parameters.get("typed_encoder_width",0):
                with torch.random.fork_rng(devices=[]):
                    self.typed_input = TypedFeatureEncoder(int(parameters["typed_encoder_width"]),width)

        def forward(self, dynamic, static):
            sequence = dynamic.transpose(1, 2)
            context = self.condition(torch.cat([static, dynamic.mean(dim=1)], dim=1)).unsqueeze(2)
            projected = self.projection(sequence) + context
            if self.typed_input is not None:
                projected = projected + self.typed_input(dynamic,static)
            hidden = self.blocks(projected)
            logits = self.head(hidden) + self.linear_skip(sequence) + self.static_skip(static).unsqueeze(2)
            if self.local_head is not None:
                logits = logits + self.local_head(dynamic, static).transpose(1, 2)
            return logits.transpose(1, 2)
    if parameters["kernel_size"] != 3 or parameters["device"] != "cpu":
        raise ValueError("This verified architecture uses kernel size 3 on CPU.")
    if parameters.get("layout", "shared") == "shared":
        return HydroTCN(len(GENERATORS))
    if parameters.get("layout") != "independent":
        raise ValueError("Unknown network layout.")
    class GeneratorTCNs(nn.Module):
        def __init__(self):
            super().__init__()
            self.networks = nn.ModuleList([HydroTCN(1) for _ in GENERATORS])
        def forward(self, dynamic, static):
            # Concatenation assigns disjoint targets; there is no prediction average.
            return torch.cat([network(dynamic, static) for network in self.networks], dim=2)
    return GeneratorTCNs()


def predict_sequence(model, dynamic, static, batch_size):
    import torch
    model.eval()
    chunks = []
    with torch.inference_mode():
        for start in range(0, len(dynamic), batch_size):
            logits = model(torch.from_numpy(dynamic[start:start+batch_size]),
                           torch.from_numpy(static[start:start+batch_size]))
            chunks.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(chunks).astype(np.float32)


def epoch_learning_rate(parameters, epoch):
    if not 1 <= epoch <= parameters["max_epochs"]:
        raise ValueError("Epoch outside the declared schedule.")
    fraction = (epoch - 1) / max(parameters["max_epochs"] - 1, 1)
    return float(parameters["learning_rate"] * (0.1 + 0.9 * (1 + np.cos(np.pi * fraction)) / 2))


## Train the shared TCN

Fit from scratch for the fixed 22 epochs. Batches are shuffled using seed 42,
and all 14 binary targets are trained jointly with binary cross-entropy.
No pretrained weights or stopping labels are read. CPU training uses four
threads and clips the gradient norm to 1.


In [10]:
def train_tcn():
    torch = initialize_torch()
    from torch import nn
    raw_d, raw_s, dn, sn = sequence_inputs(train_cases)
    eval_d, eval_s, edn, esn = sequence_inputs(prediction_metadata(test_cases))
    if dn != DYNAMIC_NAMES or sn != STATIC_NAMES or edn != dn or esn != sn:
        raise ValueError("Feature schema differs between training and prediction.")
    scaler = fit_scaler(raw_d, raw_s, BASE_PARAMETERS["normalization_clip"])
    fit_d, fit_s = scale_inputs(raw_d, raw_s, scaler)
    val_d, val_s = scale_inputs(eval_d, eval_s, scaler)
    del raw_d, raw_s, eval_d, eval_s
    y = torch.from_numpy(sequence_labels(train_cases))
    model = make_model(BASE_PARAMETERS)
    parameter_count = sum(p.numel() for p in model.parameters())
    if parameter_count != 368506:
        raise ValueError("Unexpected TCN architecture.")
    batch_generator = torch.Generator().manual_seed(SEED)
    optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_PARAMETERS["learning_rate"],
                                  weight_decay=BASE_PARAMETERS["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    print(f"Training one network with {parameter_count:,} parameters on {len(train_cases)} cases", flush=True)
    history = []
    for epoch in range(1, EPOCHS + 1):
        tick = perf_counter()
        lr = epoch_learning_rate(BASE_PARAMETERS, epoch)
        for group in optimizer.param_groups:
            group["lr"] = lr
        model.train()
        order = torch.randperm(len(train_cases), generator=batch_generator).numpy()
        accumulated_loss = 0.0
        for start in range(0, len(order), BASE_PARAMETERS["batch_size"]):
            indices = order[start:start + BASE_PARAMETERS["batch_size"]]
            dynamic = torch.from_numpy(fit_d[indices])
            static = torch.from_numpy(fit_s[indices])
            optimizer.zero_grad(set_to_none=True)
            logits = model(dynamic, static)
            loss = criterion(logits, y[indices])
            if not torch.isfinite(loss):
                raise ValueError("Nonfinite training loss.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), BASE_PARAMETERS["gradient_clip"])
            optimizer.step()
            accumulated_loss += float(loss.detach()) * len(indices)
        history.append({"epoch": epoch, "learning_rate": lr,
                        "training_logloss": accumulated_loss / len(train_cases)})
        print(f"Epoch {epoch:02d}/{EPOCHS}: loss={history[-1]['training_logloss']:.5f}, "
              f"lr={lr:.7f}, {perf_counter() - tick:.1f}s", flush=True)
    probability = predict_sequence(model, val_d, val_s, BASE_PARAMETERS["batch_size"])
    del model, optimizer, fit_d, fit_s, val_d, val_s, y, logits, loss
    gc.collect()
    return {g: probability[:, :, j] for j, g in enumerate(GENERATORS)}, pd.DataFrame(history)

predictions, training_history = train_tcn()


Training one network with 368,506 parameters on 2922 cases
Epoch 01/22: loss=0.19579, lr=0.0005000, 20.6s
Epoch 02/22: loss=0.12401, lr=0.0004999, 20.1s
Epoch 03/22: loss=0.11154, lr=0.0004995, 20.3s
Epoch 04/22: loss=0.10433, lr=0.0004990, 20.9s
Epoch 05/22: loss=0.09935, lr=0.0004982, 20.5s
Epoch 06/22: loss=0.09529, lr=0.0004972, 20.0s
Epoch 07/22: loss=0.09134, lr=0.0004959, 21.1s
Epoch 08/22: loss=0.08864, lr=0.0004945, 20.6s
Epoch 09/22: loss=0.08519, lr=0.0004928, 20.0s
Epoch 10/22: loss=0.08327, lr=0.0004909, 21.3s
Epoch 11/22: loss=0.08056, lr=0.0004888, 32.9s
Epoch 12/22: loss=0.07868, lr=0.0004864, 23.2s
Epoch 13/22: loss=0.07657, lr=0.0004839, 22.0s
Epoch 14/22: loss=0.07465, lr=0.0004811, 23.3s
Epoch 15/22: loss=0.07289, lr=0.0004782, 48.0s
Epoch 16/22: loss=0.07065, lr=0.0004750, 53.8s
Epoch 17/22: loss=0.06964, lr=0.0004716, 42.8s
Epoch 18/22: loss=0.06789, lr=0.0004680, 41.5s
Epoch 19/22: loss=0.06669, lr=0.0004643, 51.8s
Epoch 20/22: loss=0.06470, lr=0.0004603, 52.1s
E

## Save the prediction CSV

Probabilities are placed in the exact case and target order from the supplied
template. The fingerprint compares the generated bytes with the retained
submission without loading that file or any trained model.


In [11]:
def export_predictions(frame, path, reference_sha256):
    """Save a complete probability table and print its reproducibility fingerprint."""
    if frame.shape != template.shape or not frame.columns.equals(template.columns):
        raise ValueError("Prediction schema differs from the supplied template.")
    np.testing.assert_array_equal(frame["Run No"], template["Run No"])
    values = frame[LABELS].to_numpy()
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Predictions must be finite probabilities in [0, 1].")
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        archive = path.parent / "archive" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        archive.mkdir(parents=True, exist_ok=False)
        shutil.copy2(path, archive / path.name)
        print("Previous output preserved:", archive / path.name)
    frame.to_csv(path, index=False)
    saved = pd.read_csv(path)
    if saved.shape != template.shape or not saved.columns.equals(template.columns):
        raise ValueError("CSV schema changed during export.")
    np.testing.assert_array_equal(saved["Run No"], template["Run No"])
    np.testing.assert_allclose(saved[LABELS], values, rtol=1e-7, atol=1e-7)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print("CSV:", path)
    print("Shape:", saved.shape)
    print("SHA256:", digest)
    print("Exact reference match:", digest == reference_sha256)
    print(f"Total elapsed: {(perf_counter() - NOTEBOOK_STARTED) / 60:.1f} minutes")
    return path

submission_path = export_predictions(assemble_predictions(test_cases, predictions), OUTPUT_PATH, EXPECTED_CSV_SHA256)


CSV: C:\Users\fredr\Documents\MIDT\TDT4173-course-project\outputs\final_notebooks\20261106T095122Z_clean\raw_only_reproduction\outputs\tcn_submission.csv
Shape: (725, 2353)
SHA256: 84a83f1df1206acfbb69b32c9ec831e4b12fa7c6c64affd9ac1bcac88613c693
Exact reference match: True
Total elapsed: 14.9 minutes
